# Create CONCYTEC/PROCIENCIA Awards (GRANT PATTERN, Peru)

Peruvian science, technology, and innovation subsidies/projects from PROCIENCIA's official Observatorio de Subvenciones:

- Source list: https://observatorio.prociencia.gob.pe/subvenciones
- Official gob.pe announcement: https://www.gob.pe/institucion/concytec/noticias/1370851-peru-abre-al-publico-la-informacion-sobre-sus-proyectos-y-resultados-financiados-con-recursos-del-estado
- S3 location: `s3a://openalex-ingest/awards/concytec_prociencia/concytec_prociencia_projects.parquet`

**Source decision:** The observatory is an official CONCYTEC/PROCIENCIA public portal. The gob.pe announcement says it gives public access to projects and results financed with Peruvian state resources from 2011 onward. This is the preferred source over PeruCRIS/general portal pages because it exposes subsidy-level detail pages with native project code, leader, executing organization, intervention type, call, dates, OCDE fields, amount in soles, and output counts.

**Funder:** `F4320326614` — Consejo Nacional de Ciencia, Tecnologia e Innovacion Tecnologica (CONCYTEC). ROR `https://ror.org/05c7j7r25`; DOI `10.13039/501100010747`.

**Expected local scrape:** 4,694 rows advertised on the 2026-05-21 source list page. The download script scans official detail pages at `/DetalleSubvencion/{id}`, preserves raw source fields as strings, and uses `prociencia-{source_detail_id}` as the stable `funder_award_id` because displayed project codes may contain promotion/variant suffixes and are not guaranteed globally unique.

**Schema notes:**
- `amount` = parsed `Monto total`; `currency` = `PEN` (source displays `S/`).
- `display_name` = project title from the detail page.
- `description` concatenates Resumen, Objetivo General, and Palabras Clave when present.
- `lead_investigator` uses the displayed `Lider de Proyecto`; names are best-effort split on the source's `FAMILY ,GIVEN` convention.
- `lead_investigator.affiliation.name` uses `Organizacion / Afiliacion` when present, otherwise the executing organization.
- `funding_type = 'research'` to match the MinCiencias/Argentina Latin American grant-pattern convention, while raw `intervention_type` remains available for QA.

**Contractor handoff:** Script/notebook are prepared locally. Contractor has no S3/Databricks access; admin must upload parquet, run this notebook, run verification, and only then wire scheduled jobs if approved.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.concytec_prociencia_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/concytec_prociencia/concytec_prociencia_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_raw_rows FROM openalex.awards.concytec_prociencia_raw;


## Step 1.5: Inspect raw + money-field scan

Mandatory grant-pattern inspection. Confirm `amount` came from the source's `Monto total` field, and that values are plausible PEN project amounts rather than years/counts.

In [ ]:
%sql
DESCRIBE openalex.awards.concytec_prociencia_raw;


In [ ]:
%sql
SELECT
    source_detail_id,
    project_code,
    display_name,
    lead_investigator_name,
    executing_organization,
    amount_text,
    amount,
    currency,
    start_date,
    end_date,
    intervention_type,
    agreement,
    call,
    landing_page_url
FROM openalex.awards.concytec_prociencia_raw
LIMIT 10;


In [ ]:
%sql
-- Money-field scan per runbook section 1.5
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'concytec_prociencia_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(AVG(TRY_CAST(amount AS DOUBLE)), 2) AS avg_amount,
    COUNT(amount) AS non_null_amount,
    SUM(CASE WHEN TRY_CAST(amount AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS positive_amount,
    COUNT(*) AS total_rows,
    collect_set(currency) AS currencies
FROM openalex.awards.concytec_prociencia_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(project_code) AS has_project_code,
    COUNT(lead_investigator_name) AS has_leader,
    COUNT(executing_organization) AS has_executing_org,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(intervention_type) AS has_intervention_type,
    COUNT(call) AS has_call,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT slug) AS distinct_slugs
FROM openalex.awards.concytec_prociencia_raw;


## Step 1.6: Fail-fast - verify CONCYTEC funder exists

The Step 2 transform joins to `openalex.common.funder`. If the funder row is absent, the join can silently emit zero rows. This must return exactly one row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320326614;


## Step 2: Transform to OpenAlex awards schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.concytec_prociencia_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320326614
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_date_cast,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_date_cast,
        COALESCE(TRY_CAST(start_year AS INT), YEAR(TRY_TO_DATE(start_date, 'yyyy-MM-dd'))) AS start_year_int,
        COALESCE(TRY_CAST(end_year AS INT), YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd'))) AS end_year_int,
        CASE
            WHEN lead_investigator_name IS NOT NULL AND INSTR(lead_investigator_name, ',') > 0
                THEN NULLIF(TRIM(try_element_at(SPLIT(lead_investigator_name, ','), 2)), '')
            ELSE NULL
        END AS leader_given_name,
        CASE
            WHEN lead_investigator_name IS NOT NULL AND INSTR(lead_investigator_name, ',') > 0
                THEN NULLIF(TRIM(try_element_at(SPLIT(lead_investigator_name, ','), 1)), '')
            ELSE NULLIF(TRIM(lead_investigator_name), '')
        END AS leader_family_name,
        NULLIF(TRIM(COALESCE(lead_investigator_affiliation, executing_organization)), '') AS leader_affiliation_name,
        CASE
            WHEN LOWER(COALESCE(executing_org_country, execution_country, '')) RLIKE 'per|peru|perú' THEN 'PE'
            ELSE NULL
        END AS leader_affiliation_country,
        CONCAT_WS('

',
            NULLIF(TRIM(summary), ''),
            CASE WHEN NULLIF(TRIM(objective), '') IS NOT NULL THEN CONCAT('Objective: ', TRIM(objective)) END,
            CASE WHEN NULLIF(TRIM(keywords), '') IS NOT NULL THEN CONCAT('Keywords: ', TRIM(keywords)) END
        ) AS award_description
    FROM openalex.awards.concytec_prociencia_raw
)
SELECT
    abs(xxhash64(CONCAT('4320326614:concytec_prociencia:', rp.slug))) % 9000000000 AS id,
    rp.display_name,
    NULLIF(rp.award_description, '') AS description,
    4320326614 AS funder_id,
    rp.slug AS funder_award_id,
    rp.amount_double AS amount,
    CASE WHEN rp.amount_double IS NOT NULL THEN 'PEN' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    COALESCE(NULLIF(rp.call, ''), NULLIF(rp.intervention_type, ''), NULLIF(rp.agreement, '')) AS funder_scheme,
    'prociencia_observatorio' AS provenance,
    rp.start_date_cast AS start_date,
    rp.end_date_cast AS end_date,
    rp.start_year_int AS start_year,
    rp.end_year_int AS end_year,
    struct(
        rp.leader_given_name AS given_name,
        rp.leader_family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            rp.leader_affiliation_name AS name,
            rp.leader_affiliation_country AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320326614:concytec_prociencia:', rp.slug))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
JOIN funder_resolved f ON f.funder_id = 4320326614
WHERE rp.slug IS NOT NULL
  AND rp.display_name IS NOT NULL
  AND rp.start_year_int IS NOT NULL;


## Step 3: Insert into `openalex_awards_raw` at priority 89

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'prociencia_observatorio' AND priority = 89;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    89 AS priority
FROM openalex.awards.concytec_prociencia_awards;


## Step 6: Verification

Grant pattern, monetary. The §6.7 amount/currency check is not waived: the official detail pages publish `Monto total` and the script maps it to PEN.

In [ ]:
%sql
SELECT COUNT(*) AS total_concytec_prociencia_awards
FROM openalex.awards.concytec_prociencia_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.concytec_prociencia_awards;


In [ ]:
%sql
-- Duplicate funder_award_id must be zero; duplicate ids must also be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.concytec_prociencia_awards;


In [ ]:
%sql
-- Data completeness / §6.7 amount check
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL OR lead_investigator.given_name IS NOT NULL THEN 1 ELSE 0 END) AS has_pi_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_pi_affiliation,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    collect_set(currency) AS currencies
FROM openalex.awards.concytec_prociencia_awards;


In [ ]:
%sql
SELECT
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    percentile_approx(amount, 0.5) AS median_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.concytec_prociencia_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS awards, ROUND(SUM(amount), 2) AS total_amount_pen
FROM openalex.awards.concytec_prociencia_awards
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 2) AS total_amount_pen
FROM openalex.awards.concytec_prociencia_awards
GROUP BY funding_type, funder_scheme
ORDER BY awards DESC
LIMIT 30;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 80) AS title,
    funder_award_id,
    ROUND(amount, 2) AS amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS pi_given,
    lead_investigator.family_name AS pi_family,
    SUBSTRING(lead_investigator.affiliation.name, 1, 60) AS affiliation,
    landing_page_url
FROM openalex.awards.concytec_prociencia_awards
ORDER BY start_year DESC, id
LIMIT 20;


In [ ]:
%sql
-- Confirm the priority insert landed as expected.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'prociencia_observatorio'
GROUP BY priority, provenance;
